In [ ]:
#import necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from skbio import DistanceMatrix
from skbio.stats.distance import permanova, anosim
from skbio.stats.ordination import pcoa
from scipy.spatial.distance import pdist, squareform
from skbio import DistanceMatrix
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import re

# Data Analysis

In [ ]:
### define setting to indicate raw data in saved files
database = "MGnify" # MGnify / IGC / Qin
grouping = "Groups" # Groups / Subgroups
new_discovery_study = "Werner" # ValdesMas / Werner
file_prefix = database +"_"+grouping + "_" + new_discovery_study + "_"
print(file_prefix)

In [ ]:
# Fig 1B (all identified metaproteins)
data = pd.read_csv(file_prefix+"all_metaproteins(all_studies_normalized_5).csv", sep=",", index_col=0)#
metaprotein_set = "all_metaproteins"
# Fig 1C + Supplementary Figure 3 (metaproteins identified in all discovery data sets)
#data = pd.read_csv(file_prefix+"shared_metaproteins(discovery_studies_normalized_5).csv", sep=",", index_col=0)#
#metaprotein_set = "shared_metaproteins"
# Fig 1F (metaproteins identified in all discovery data sets after batch correction with MMUPHIN)
#data = pd.read_csv(file_prefix+"shared_metaproteins(discovery_studies_normalized_MMUPHIN_corrected).csv", sep=",", index_col=0)#
#metaprotein_set = "shared_metaproteins_MMUPHIN_corrected"

# Fig 1G (metaproteins from 1F with variance explained by IBD > 0.2)
### read input from Variance analysis and only use the metaproteins provided by this file
#variance_analysis_output = pd.read_csv("./NotebooksForUpload/Qin_Subgroups_Werner_variance_explained_shared_MMUPHin.csv", sep=",", index_col=0)#
#selected_proteins = list(variance_analysis_output["species"])
#data = data.loc[selected_proteins,:]
#metaprotein_set = "selected_metaproteins_MMUPHIN_corrected"


#data.rename(columns=lambda s: s.replace("P36_UCa", "P35_UCa"), inplace=True)

data.head(5)

allMetaproteins = list(data.index)


In [ ]:
### load metadata
meta_data_path = r"SupplementaryFile1(WernerDiscovery)-Revision.xlsx"
meta_data = pd.read_excel(meta_data_path, sheet_name="SampleMetadata", index_col = "SampleID" )
print(meta_data.head(5))

### to make matching of sample names from meta data and mearurement data easier, replace all special characters by "_"
meta_data.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in meta_data.columns]
data.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in data.columns]


compare_panel = data
### join sample annotation and metaproteni abundance
for measurementID in compare_panel.columns:
    #print(type(measurementID))
    for sampleID in meta_data.columns:
        if sampleID in measurementID:
            compare_panel.loc["condition", measurementID]= meta_data.loc["condition", sampleID]
            compare_panel.loc["disease", measurementID]= meta_data.loc["disease", sampleID]
            compare_panel.loc["study", measurementID]= meta_data.loc["study", sampleID]
            compare_panel.loc["UseCase", measurementID]= meta_data.loc["UseCase", sampleID]
compare_panel

In [ ]:
# Filter data based on use case (discovery, validation, specificity)
data = compare_panel
row = data.loc["UseCase"]

discovery_columns = row[row == 'BiomarkerDiscovery'].index
validation_columns = row[row == 'BiomarkerValidation'].index
specificity_columns = row[row == 'DiseaseSpecificity'].index

df_discovery = data[discovery_columns]
df_validation = data[validation_columns]
df_specificity = data[specificity_columns]

UseCaseSpecData={"discovery":df_discovery,"validation":df_validation,"specificity":df_specificity}
print(len(df_discovery.columns)) # discovery sample = 120


### Mulitvariate Analysis

In [ ]:
scaler = MinMaxScaler()
metaproteins_to_use = allMetaproteins  # allMetaproteins / monitoring / diagnosis / list(set(monitoring + diagnosis))
df = df_discovery# use case data to visualize


data=df.T
#data = df.loc[metaproteins_to_use,:].T # if no filtering applied for metaproteins: data=df.T
metadata = df.loc[["UseCase","study","disease","condition"]]
#metadata = df.iloc[:5]
#metadata = df.iloc[:-5]


# Define colors based on categories (face and edge color will be the same)
category_colors = {
    "Lehmann": "#4472c4",
    "Thuy-Boun": "#33a02c",
    "Henry": "#ff0000",
    "Lloyd-Price": "#cab2d6",
    "Werner": "#fcca27",
    "Valdes-Mas(ISR)": "#fcca27",
    "Wolf_2021":"yellow",
    "Wolf_2024_BL-R":"orange",
    "Wolf_2024_BL-A":"orange",
    "Wolf_2024_BL-M":"orange",
    "Wolf_2024_W14-M":"orange",
    "Wolf_2024_W14-R":"green",
    "Wolf_2024_W14-A":"red"
}

# Mapping dictionaries to standardize metadata
color_map = {
    "Lehmann": "#4472c4",
    "Thuy-Boun": "#33a02c",
    "Henry": "#ff0000",
    "Lloyd-Price": "#cab2d6",
    "Werner": "#fcca27",
    "Valdes-Mas(ISR)": "#fcca27",
    "Wolf_2021":"yellow",
    "Wolf_2024_BL-A":"orange",
    "Wolf_2024_BL-R":"orange",
    "Wolf_2024_W14-R":"green",
    "Wolf_2024_W14-A":"red"
}
shape_map = {'active': 'circle', 'remission': 'triangle', '0':'circle'}
fill_map = {'diseased': 'filled', 'control': 'not_filled'}

filled_condition = data['condition'] == 'diseased'

colors = []

#
for sample in data.index:
    category = data.loc[sample,"study"]
    color = category_colors[category]
    colors.append(color)
    
#data = data.drop(["study","condition","disease","disease_state","UseCase"],axis=1)

data = data.drop(["study","condition","disease","UseCase"],axis=1)
data = data.loc[:,metaproteins_to_use]

print(type(data.iloc[2,3]))


# Convert to a DataFrame
abundance_df = pd.DataFrame(data).astype(np.float64)
print(abundance_df)

df_scaled = scaler.fit_transform(abundance_df.to_numpy())
abundance_df = pd.DataFrame(df_scaled, columns=abundance_df.columns)

# Calculate the distance matrix
distance_matrix = pdist(abundance_df, metric='braycurtis')
distance_matrix = squareform(distance_matrix)  # Convert to square form

# Convert the distance matrix into scikit-bio's DistanceMatrix format
skbio_distance_matrix = DistanceMatrix(distance_matrix, ids=data.index)




In [ ]:
### PERMANOVA
# compute the PERMANOVA p-values from skbio_distance_matrix, based on the differences between the disease groups (IBD vs. healthy)
# and the different studies
dm = skbio_distance_matrix

grouping1 = metadata.loc["study"]
#grouping2 = metadata.loc["disease"]
grouping2 = metadata.loc["condition"]

permanovaStudy = permanova(dm, grouping1, permutations=999)
permanovaDisease = permanova(dm, grouping2, permutations=999)
print("Permanova study: "+ str(permanovaStudy))
print("Permanova disease "+ str(permanovaDisease))

### export results for Supplementary Figure
filename= f"Permanova_Results_Revision_{file_prefix}_{metaprotein_set}.txt"
f = open(filename, "a")
f.write("Permanova (Study)\n")
f.write(str(permanovaStudy))
f.write("\n")
f.write("Permanova (Disease)\n")
f.write(str(permanovaDisease))
f.close()

In [ ]:
from adjustText import adjust_text
# Perform PCoA
pcoa_results = pcoa(skbio_distance_matrix)

# Create a DataFrame to store the principal coordinates
df_pcoa = pd.DataFrame(pcoa_results.samples.values[0:len(pcoa_results.samples.values),0:2], columns=['PC1', 'PC2'], index=pcoa_results.samples.index)
print("\nPCoA Results:\n", df_pcoa)


# Plot the results (PC1 vs PC2), with face and edge colors based on category and fill condition
fig, ax = plt.subplots()

# Plot each point with its corresponding color and fill style
for i, sample in enumerate(df_pcoa.index):
    facecolor = colors[i] if filled_condition[sample] else 'none'
    edgecolor = colors[i]  # Edgecolor is always the same as facecolor
    
    ax.scatter(df_pcoa['PC1'][i], df_pcoa['PC2'][i],
                edgecolor=edgecolor, facecolor=facecolor, s=50, marker='o')

# Label each point with its sample name
#for i, label in enumerate(df_pcoa.index):
#    plt.text(df_pcoa['PC1'][i], df_pcoa['PC2'][i], label, fontsize=6, ha='right')


# legend 
# Legend 1: Disease condition (control vs diseased)
legend_elements_condition = [
    Line2D([0], [0], marker='o', color='w', label='Control (unfilled)',
           markerfacecolor='none', markeredgecolor='black', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Diseased (filled)',
           markerfacecolor='black', markeredgecolor='black', markersize=10)
]

# Legend 2: Sample categories
legend_elements_category = [
    Line2D([0], [0], marker='o', color='w', label='Lehmann', markerfacecolor='#4472c4', markeredgecolor='#4472c4', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Thuy-Boun', markerfacecolor='#33a02c', markeredgecolor='#33a02c', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Henry', markerfacecolor='#ff0000', markeredgecolor='#ff0000', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Lloyd-Price', markerfacecolor='#cab2d6', markeredgecolor='#cab2d6', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Werner', markerfacecolor='#fcca27', markeredgecolor='#fcca27', markersize=10)
]

#for i in strongest_var_indices2:
#    plt.arrow(0, 0, correlation_matrix[i, 0], correlation_matrix[i, 1], color='red', alpha=0.7, 
#              head_width=0.03, head_length=0.03)
#    plt.text(correlation_matrix[i, 0] * 1.1, correlation_matrix[i, 1] * 1.1, 
#             abundance_df.columns[i], color='red', fontsize=7)
texts = []

#for i in strongest_var_indices2:
#    plt.arrow(0, 0, correlation_matrix[i, 0], correlation_matrix[i, 1],
#              color='red', alpha=0.7, head_width=0.03, head_length=0.03)
#    texts.append(
#        plt.text(correlation_matrix[i, 0] * 1.1,
#                 correlation_matrix[i, 1] * 1.1,
#                 abundance_df.columns[i].replace("Meta-Protein ",""),
#                 color='red', fontsize=7)
#    )

# Automatically adjust text positions to avoid overlap
#adjust_text(texts, expand_text=(1.2, 1.4), force_text=0.1)

#plt.legend(handles=legend_elements_condition, title='Condition', loc='upper left',bbox_to_anchor=(0.0, 1) )#bbox_to_anchor=(1.35, 1)
#plt.legend(handles=legend_elements_category, title='Study', loc='upper right',bbox_to_anchor=(1, 1))

handles = legend_elements_condition + legend_elements_category
labels = [legend.get_label() for legend in handles]

plt.legend(handles=handles, labels=labels, title='Condition and Study', loc='upper left', bbox_to_anchor=(1.05, 1.0))#bbox_to_anchor=(0.0, 1)
ax.set_ylim(-0.4,0.4) # -0.4,0.4

plt.title('PCoA (Relative Protein Abundance)')
plt.xlabel(f'Principal Coordinate 1 ({pcoa_results.proportion_explained[0]:.2%} variance)')
plt.ylabel(f'Principal Coordinate 2 ({pcoa_results.proportion_explained[1]:.2%} variance)')
plt.tight_layout() 
#### plot the significances between the disease groups and studies from permanova

plt.grid(True)
plt.savefig(file_prefix+metaprotein_set+"(PCoA_5).png", dpi=300, bbox_inches='tight')  # dpi controls the resolution

plt.show()

In [ ]:
### Supplementary Figrue 3 (use: shared protein groups identified in all studies)
from adjustText import adjust_text

principal_coords = df_pcoa  # Get the first 2 dimensions (axes)

# Calculate the correlation between the original variables and the first two principal coordinates
correlation_matrix = np.corrcoef(abundance_df.T, principal_coords.T)[:abundance_df.shape[1], abundance_df.shape[1]:]
# We will filter the variables with the highest absolute correlation with any of the two dimensions
strongest_vars_threshold = 0.58  # Set a threshold, e.g., abs(correlation) > 0.6
strongest_var_indices1 = np.where(np.abs(correlation_matrix[:, 0]) > strongest_vars_threshold)[0]
strongest_var_indices2 = np.where(np.abs(correlation_matrix[:, 1]) > strongest_vars_threshold)[0]


# Perform PCoA
pcoa_results = pcoa(skbio_distance_matrix)

# Create a DataFrame to store the principal coordinates
df_pcoa = pd.DataFrame(pcoa_results.samples.values[0:len(pcoa_results.samples.values),0:2], columns=['PC1', 'PC2'], index=pcoa_results.samples.index)
print("\nPCoA Results:\n", df_pcoa)


# Plot the results (PC1 vs PC2), with face and edge colors based on category and fill condition
fig, ax = plt.subplots()

# Plot each point with its corresponding color and fill style
for i, sample in enumerate(df_pcoa.index):
    facecolor = colors[i] if filled_condition[sample] else 'none'
    edgecolor = colors[i]  # Edgecolor is always the same as facecolor
    
    ax.scatter(df_pcoa['PC1'][i], df_pcoa['PC2'][i],
                edgecolor=edgecolor, facecolor=facecolor, s=50, marker='o')



# legend 
# Legend 1: Disease condition (control vs diseased)
legend_elements_condition = [
    Line2D([0], [0], marker='o', color='w', label='Control (unfilled)',
           markerfacecolor='none', markeredgecolor='black', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Diseased (filled)',
           markerfacecolor='black', markeredgecolor='black', markersize=10)
]

# Legend 2: Sample categories
legend_elements_category = [
    Line2D([0], [0], marker='o', color='w', label='Lehmann', markerfacecolor='#4472c4', markeredgecolor='#4472c4', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Thuy-Boun', markerfacecolor='#33a02c', markeredgecolor='#33a02c', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Henry', markerfacecolor='#ff0000', markeredgecolor='#ff0000', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Lloyd-Price', markerfacecolor='#cab2d6', markeredgecolor='#cab2d6', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Werner', markerfacecolor='#fcca27', markeredgecolor='#fcca27', markersize=10)

]
texts = []

for i in strongest_var_indices2:
    plt.arrow(0, 0, correlation_matrix[i, 0]*0.7, correlation_matrix[i, 1]*0.7,
              color='red', alpha=0.7, head_width=0.03, head_length=0.03)
    texts.append(
        plt.text(correlation_matrix[i, 0]*1.01, #*1.3
                 correlation_matrix[i, 1]*1.01,
                 abundance_df.columns[i],
                 color='red', fontsize=6)
    )

# Automatically adjust text positions to avoid overlap
adjust_text(texts, expand_text=(1.2, 1.4), force_text=1.9)

handles = legend_elements_condition + legend_elements_category
labels = [legend.get_label() for legend in handles]

plt.legend(handles=handles, labels=labels, title='Condition and Study', loc='upper left', bbox_to_anchor=(1.05, 1.0))#bbox_to_anchor=(0.0, 1)
ax.set_ylim(-0.65,0.65) # -0.4,0.4

plt.title('PCoA (Relative Protein Abundance)')
plt.xlabel(f'Principal Coordinate 1 ({pcoa_results.proportion_explained[0]:.2%} variance)')
plt.ylabel(f'Principal Coordinate 2 ({pcoa_results.proportion_explained[1]:.2%} variance)')
plt.tight_layout() 

plt.grid(True)
plt.savefig("SupplementaryFigure3", dpi=300, bbox_inches='tight')  # dpi controls the resolution

plt.show()